# Cookie Cats A/B Test: Gate at Level 30 vs Level 40

**Question:** Does moving the first progress gate from level 30 (`gate_30`, control) to level 40 (`gate_40`, treatment) change player retention and engagement?

- **Primary metric:** `retention_7`
- **Secondary metrics:** `retention_1`, `sum_gamerounds`
- **Settings:** α = 0.05, two-sided tests, 95% confidence intervals, intention-to-treat (all randomized players)

## Part 1 – Setup
### 1.1 Configuration
Edit `DATA_PATH` if your file sits somewhere else in Google Drive.

In [ ]:
# ---- Edit this path if needed ----
DATA_PATH = "/content/drive/MyDrive/cookie_cats.csv"

# Experiment settings
CONTROL, TREATMENT = "gate_30", "gate_40"
ALPHA = 0.05          # significance level (two-sided)
CONF_LEVEL = 0.95     # confidence level for all intervals
SEED = 42             # seed for bootstrap and permutation steps
N_BOOT = 5_000        # bootstrap / permutation resamples

### 1.2 Imports and plot style

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility: one seeded generator for all resampling
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Plot style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 100,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})
PALETTE = {CONTROL: "#1f77b4", TREATMENT: "#ff7f0e"}

# Table display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")
print(f"Seed = {SEED}, alpha = {ALPHA}, bootstrap resamples = {N_BOOT:,}")

### 1.3 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.4 Load the data

In [ ]:
def load_data(path):
    """Load the CSV; print a clear message and stop if the path is wrong."""
    if not os.path.isfile(path):
        print(f"ERROR: file not found at:\n   {path}")
        print("Check DATA_PATH in cell 1.1 (spelling, folder, and that Drive is mounted).")
        # Help locate the file: list CSVs in the top level of My Drive
        candidates = sorted(glob.glob("/content/drive/MyDrive/*.csv"))
        if candidates:
            print("\nCSV files found in My Drive:")
            for c in candidates[:20]:
                print("   ", c)
        raise FileNotFoundError(path)
    return pd.read_csv(path)

df = load_data(DATA_PATH)

print(f"Loaded {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Quick check against the expected size
EXPECTED_SHAPE = (90_189, 5)
if df.shape == EXPECTED_SHAPE:
    print("Shape matches the expected 90,189 x 5.")
else:
    print(f"Warning: expected {EXPECTED_SHAPE}, got {df.shape}. Confirm this is the right file.")

df.head()

## Part 2 – Data Integrity
### 2.1 Column types and a quality log
Every check below writes one line to a data quality log, printed at the end of the part.

In [ ]:
# Data quality log: one row per check
dq_log = []

def log_check(check, passed, detail):
    """Record a data quality check and print it."""
    status = "PASS" if passed else "FLAG"
    dq_log.append({"check": check, "status": status, "detail": detail})
    print(f"[{status}] {check}: {detail}")

print("Column types as loaded:")
print(df.dtypes.to_string())

### 2.2 Convert retention columns to boolean
`pd.read_csv` usually reads `TRUE`/`FALSE` as booleans already. If not (for example strings or 0/1), this cell converts them and flags any value it cannot map.

In [ ]:
BOOL_MAP = {
    "true": True, "false": False,
    "t": True, "f": False,
    "yes": True, "no": False,
    "1": True, "0": False,
}

def to_bool(series):
    """Convert a column to bool; return (converted, number of unmapped values)."""
    if pd.api.types.is_bool_dtype(series):
        return series, 0
    mapped = series.astype(str).str.strip().str.lower().map(BOOL_MAP)
    n_bad = int(mapped.isna().sum() - series.isna().sum())
    return mapped, n_bad

for col in ["retention_1", "retention_7"]:
    original_dtype = df[col].dtype
    converted, n_bad = to_bool(df[col])
    if n_bad == 0 and converted.notna().all():
        df[col] = converted.astype(bool)
        note = "already boolean" if original_dtype == bool else f"converted from {original_dtype}"
        log_check(f"{col} is boolean", True, note)
    else:
        df[col] = converted  # keep NaN for unmapped values so they show up as missing
        log_check(f"{col} is boolean", False, f"{n_bad:,} values could not be mapped to True/False")

### 2.3 Unique IDs, missing values, duplicates

In [ ]:
# userid uniqueness
n_ids = df["userid"].nunique()
n_dup_ids = len(df) - n_ids
log_check("userid is unique", n_dup_ids == 0,
          f"{n_ids:,} unique IDs across {len(df):,} rows ({n_dup_ids:,} repeated)")

# Missing values per column
missing = df.isna().sum()
n_missing = int(missing.sum())
detail = "no missing values" if n_missing == 0 else \
    ", ".join(f"{c}={m:,}" for c, m in missing.items() if m > 0)
log_check("No missing values", n_missing == 0, detail)

# Fully duplicated rows
n_dup_rows = int(df.duplicated().sum())
log_check("No duplicated rows", n_dup_rows == 0, f"{n_dup_rows:,} fully duplicated rows")

# A repeated userid would mean a player was counted twice (possibly in both groups)
if n_dup_ids > 0:
    dup_ids = df.loc[df["userid"].duplicated(keep=False), "userid"]
    in_both = df[df["userid"].isin(dup_ids)].groupby("userid")["version"].nunique().gt(1).sum()
    print(f"   Note: {in_both:,} repeated IDs appear in both groups. Review before analysis.")

### 2.4 Value checks on `version` and `sum_gamerounds`

In [ ]:
# version must contain exactly the two expected arms
expected_versions = {CONTROL, TREATMENT}
found_versions = set(df["version"].dropna().unique())
unexpected = found_versions - expected_versions
log_check("version has only gate_30 / gate_40", not unexpected and found_versions == expected_versions,
          f"found {sorted(found_versions)}" + (f"; unexpected {sorted(unexpected)}" if unexpected else ""))

# sum_gamerounds must be whole, non-negative numbers
rounds = df["sum_gamerounds"]
is_int = pd.api.types.is_integer_dtype(rounds)
log_check("sum_gamerounds is integer", is_int, f"dtype = {rounds.dtype}")

n_negative = int((rounds < 0).sum())
log_check("sum_gamerounds is non-negative", n_negative == 0, f"{n_negative:,} negative values")

# Zero-round players are kept (they were randomized); just report them
n_zero = int((rounds == 0).sum())
log_check("Zero-round players kept", True,
          f"{n_zero:,} players ({n_zero / len(df):.2%}) played 0 rounds")

# Extreme values: compare the maximum to the next highest value
top_two = rounds.nlargest(2).tolist()
ratio = top_two[0] / top_two[1] if top_two[1] > 0 else np.inf
log_check("No extreme outlier in sum_gamerounds", ratio < 5,
          f"max = {top_two[0]:,}, next highest = {top_two[1]:,} ({ratio:.1f}x); handled in Part 5")

print("\nsum_gamerounds summary:")
print(rounds.describe(percentiles=[0.25, 0.5, 0.75, 0.99, 0.999]).round(1).to_string())

### 2.5 Data quality log

In [ ]:
dq_table = pd.DataFrame(dq_log)
n_flags = (dq_table["status"] == "FLAG").sum()

print(f"Data quality log: {len(dq_table)} checks, {n_flags} flagged\n")
display(dq_table)

if n_flags == 0:
    print("All checks passed. The data is ready for analysis.")
else:
    flagged = ", ".join(dq_table.loc[dq_table["status"] == "FLAG", "check"])
    print(f"Flagged: {flagged}.")
    print("No rows are removed here. The primary analysis keeps every randomized player; "
          "flagged items are handled explicitly in later parts.")